# EOG shuffled-pairing control

This analysis compares the matched joint hEOG/vEOG regression model with a
shuffled-pairing control.

For the EOG model, each EEG target trial is paired with hEOG/vEOG predictors
from the same trial.

For the shuffled-pairing model, hEOG/vEOG predictors are taken from a
different trial of the same participant. This shuffled pairing is applied
during both training and held-out prediction. The resulting predictions are
evaluated against the EEG targets from the original held-out trials.

Both models use the same participants, EEG channels, cross-validation
partitions, target trials, and target-scoring procedure.

The primary comparison is therefore:

- EOG model
- Shuffled-pairing model, original-target scoring

The participant is the population-level inferential unit. Trial-level
differences are first calculated within each participant and then summarized
across participants.

## 1. Environment Setup and Imports

Import the project result-loading, statistical, and plotting functions.

In [ ]:
# %%
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests

SOURCE_ROOT = Path(r"D:\Thesis\Codes\Source")

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Source directory not found: {SOURCE_ROOT}"
    )

if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from face2eeg.core.results import load_experiment_result

from face2eeg.core.stats import (
    summarize_trials,
    compare_methods_trial_level,
    fisher_z,
    inv_fisher_z,
    sign_flip_pvalue,
)

from face2eeg.analysis import validate_trial_alignment
from face2eeg.plotting import plot_eog_shuffled_pairing_summary
from face2eeg.core.config import VALID_SUBJECTS, SEED

%matplotlib inline

## 2. Analysis Configuration

Define the matched and shuffled-pairing result directories, participants,
EEG channels, sampling parameters, and statistical settings.

In [ ]:
# %%
BASE = Path(r"D:\Thesis\Codes\Data\Experiments\Cross-Validation")

MATCHED_DIR = BASE / "EOG-Regression" / "Results"
SHUFFLED_DIR = BASE / "EOG-Shuffled" / "Results"

FIG_DIR = SHUFFLED_DIR / "Analysis-Figures"
TABLE_DIR = SHUFFLED_DIR / "Analysis-Tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Participants and EEG channels
# ---------------------------------------------------------------------

SUBJECTS = list(VALID_SUBJECTS)

EXPECTED_SUBJECTS = [1, 2, 4, 6, 7, 8, 9, 10, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22]

CHANNELS = ["Fz", "Pz", "Oz"]


# ---------------------------------------------------------------------
# Experimental parameters
# ---------------------------------------------------------------------

N_TRIALS = 40
N_FOLDS = 5
N_TRAIN_TRIALS = 32
N_TEST_TRIALS = 8

EEG_FS = 128
TRIAL_DURATION = 60

EXPECTED_TRIAL_SAMPLES = EEG_FS * TRIAL_DURATION

EOG_REFERENCES = ["heog", "veog"]
EOG_FILTER_BAND = [4.0, 45.0]

SHUFFLE_SEED = int(SEED)


# ---------------------------------------------------------------------
# Statistical parameters
# ---------------------------------------------------------------------

N_BOOT = 10_000
N_PERM = 10_000
CI_LEVEL = 0.95


# ---------------------------------------------------------------------
# Conditions
# ---------------------------------------------------------------------

EOG_CONDITION = "EOG model"

SHUFFLED_CONDITION = "Shuffled-pairing model, original-target scoring"

CONDITIONS = [EOG_CONDITION, SHUFFLED_CONDITION]

PLOT_CONDITIONS = (EOG_CONDITION, SHUFFLED_CONDITION)

PLOT_LABELS = ("EOG model", "Shuffled (orig. target)")


# ---------------------------------------------------------------------
# Configuration checks
# ---------------------------------------------------------------------

assert SUBJECTS == EXPECTED_SUBJECTS
assert len(SUBJECTS) == 18
assert len(CHANNELS) == 3
assert N_TRIALS == N_FOLDS * N_TEST_TRIALS
assert N_TRAIN_TRIALS + N_TEST_TRIALS == N_TRIALS
assert EXPECTED_TRIAL_SAMPLES == 7680

for directory in (MATCHED_DIR, SHUFFLED_DIR):
    if not directory.is_dir():
        raise FileNotFoundError(f"Results directory not found: {directory}")


print("Participants  :", len(SUBJECTS))
print("Channels      :", CHANNELS)
print("Trials        :", N_TRIALS)
print("CV folds      :", N_FOLDS)
print("Train/test    :", N_TRAIN_TRIALS, "/", N_TEST_TRIALS)
print("Trial length  :", TRIAL_DURATION, "seconds")
print("EEG rate      :", EEG_FS, "Hz")
print("Samples/trial :", EXPECTED_TRIAL_SAMPLES)
print("EOG channels  :", EOG_REFERENCES)
print("EOG filter    :", EOG_FILTER_BAND)
print("Shuffle seed  :", SHUFFLE_SEED)
print("Bootstrap     :", N_BOOT)
print("Sign-flip     :", N_PERM)
print("Matched dir   :", MATCHED_DIR)
print("Shuffled dir  :", SHUFFLED_DIR)

## 3. Input-File Audit

Verify that the matched EOG and shuffled-pairing result files exist for
every participant and each of the three EEG channels.

A complete analysis requires 54 EOG-model and 54 shuffled-pairing
result files.

In [ ]:
# %%
file_map = {channel: {} for channel in CHANNELS}
missing_files = []

for channel in CHANNELS:
    for subject in SUBJECTS:
        matched_path = MATCHED_DIR / f"sub{subject:02d}_ch{channel}.pkl"
        shuffled_path = SHUFFLED_DIR / f"sub{subject:02d}_ch{channel}.pkl"
        file_map[channel][subject] = {"matched": matched_path, "shuffled": shuffled_path}
        if not matched_path.is_file():
            missing_files.append(str(matched_path))
        if not shuffled_path.is_file():
            missing_files.append(str(shuffled_path))

if missing_files:
    raise FileNotFoundError(f"{len(missing_files)} result file(s) are missing:\n" + "\n".join(missing_files))

n_expected_files = len(SUBJECTS) * len(CHANNELS)

print(f"EOG model files      : {n_expected_files}")
print(f"Shuffled-pairing files: {n_expected_files}")
print("All required input files exist.")

## 4. Metadata and Result-Structure Audit

Verify the metadata and fold structure of the EOG model and
shuffled-pairing results.

The shuffled-pairing result must identify the control as
`shuffled_pairing`, indicate that shuffled pairing was enabled, and
contain the expected two EOG references and 4–45 Hz EOG filter settings.

In [ ]:
# %%
def check_result_metadata(result, subject, channel, condition):
    """Validate common result metadata and held-out fold arrays."""

    if result.subject != subject:
        raise ValueError(f"{condition}: participant mismatch.")

    if result.channel != channel:
        raise ValueError(f"{condition}: channel mismatch.")

    if result.time_window is None or float(result.time_window) != float(TRIAL_DURATION):
        raise ValueError(f"{condition}: expected time_window={TRIAL_DURATION}.")

    if result.n_splits != N_FOLDS:
        raise ValueError(f"{condition}: expected {N_FOLDS} folds.")

    expected_folds = list(range(N_FOLDS))
    actual_folds = list(result.fold_ids())

    if actual_folds != expected_folds:
        raise ValueError(f"{condition}: unexpected folds {actual_folds}.")

    for fold_idx in expected_folds:
        fold = result.folds[fold_idx]

        for key in ("y_test", "y_pred", "trials"):
            if key not in fold:
                raise KeyError(f"{condition}, fold {fold_idx}: missing {key}.")

        expected_shape = (N_TEST_TRIALS, EXPECTED_TRIAL_SAMPLES)

        if np.asarray(fold["y_test"]).shape != expected_shape:
            raise ValueError(
                f"{condition}, fold {fold_idx}: invalid y_test shape "
                f"{np.asarray(fold['y_test']).shape}."
            )

        if np.asarray(fold["y_pred"]).shape != expected_shape:
            raise ValueError(
                f"{condition}, fold {fold_idx}: invalid y_pred shape "
                f"{np.asarray(fold['y_pred']).shape}."
            )


def check_shuffled_config(config):
    """Validate the expected shuffled-pairing experiment configuration."""

    expected_values = {
        "method": "LinearRegression",
        "control": "shuffled_pairing",
        "shuffle_pairing": True,
        "shuffle_seed": SHUFFLE_SEED,
        "n_points": N_TRIALS,
        "n_splits": N_FOLDS,
        "no_fixed_points": True,
    }

    for key, expected in expected_values.items():
        actual = config.get(key)

        if actual != expected:
            raise ValueError(
                f"Shuffled-pairing configuration error for {key}: "
                f"expected {expected!r}, received {actual!r}."
            )

    actual_references = [str(reference).lower() for reference in config.get("references", [])]
    expected_references = [reference.lower() for reference in EOG_REFERENCES]

    if actual_references != expected_references:
        raise ValueError(
            "Unexpected EOG references: "
            f"{actual_references}; expected {expected_references}."
        )

    actual_band = np.asarray(config.get("eog_filter_band_hz", []), dtype=float)
    expected_band = np.asarray(EOG_FILTER_BAND, dtype=float)

    if actual_band.shape != expected_band.shape or not np.allclose(actual_band, expected_band):
        raise ValueError(
            "Unexpected EOG filter band: "
            f"{actual_band}; expected {expected_band}."
        )


# Load one representative result from each condition.
subject0 = SUBJECTS[0]
channel0 = CHANNELS[0]

audit_eog = load_experiment_result(str(file_map[channel0][subject0]["matched"]))
audit_shuffled = load_experiment_result(str(file_map[channel0][subject0]["shuffled"]))

check_result_metadata(audit_eog, subject0, channel0, EOG_CONDITION)
check_result_metadata(audit_shuffled, subject0, channel0, SHUFFLED_CONDITION)
check_shuffled_config(audit_shuffled.config)

print("EOG model:")
print(audit_eog.summary())

print("\nShuffled-pairing model:")
print(audit_shuffled.summary())

print("\nShuffled-pairing configuration:")
for key in (
    "method",
    "control",
    "shuffle_pairing",
    "shuffle_seed",
    "n_points",
    "n_splits",
    "references",
    "eog_filter_band_hz",
    "no_fixed_points",
):
    print(f"  {key}: {audit_shuffled.config.get(key)}")

print("\nMetadata and result structure verified.")

## 5. Pairing, Cross-Validation, and Target-Alignment QC

This section performs the main internal quality-control checks.

For every participant, channel, and fold, it verifies:

- Training and testing partitions are disjoint.
- Each fold contains all 40 trials exactly once across its partitions.
- Training EOG sources belong to the training partition.
- Testing EOG sources belong to the held-out partition.
- Neither the training nor testing mappings contains fixed points.
- Stored source-trial mappings agree with their permutations.
- Fold-specific random seeds match the training implementation.
- Matched and shuffled-pairing models use identical held-out EEG target
  trial IDs and target values.

For the shuffled-pairing model, the predictor trial is deliberately
different from the corresponding EEG target trial. The target remains
the original trial's EEG signal.

The matched and shuffled-pairing conditions are therefore evaluated
against the same original held-out EEG targets.

In [ ]:
# %%
def sort_trials_by_id(trial_ids, *arrays):
    """
    Sort trial IDs and all associated arrays together.

    This preserves the correspondence between each EEG target trial
    and its prediction.
    """
    trial_ids = np.asarray(trial_ids, dtype=int).reshape(-1)

    if np.unique(trial_ids).size != trial_ids.size:
        raise ValueError("Duplicate trial IDs detected.")

    order = np.argsort(trial_ids)
    sorted_arrays = [np.asarray(array)[order] for array in arrays]

    return (trial_ids[order], *sorted_arrays)


def check_trial_array(array, label, subject, channel):
    """Check the shape and numerical validity of trial arrays."""

    array = np.asarray(array)
    expected_shape = (N_TRIALS, EXPECTED_TRIAL_SAMPLES)

    if array.shape != expected_shape:
        raise ValueError(
            f"S{subject:02d}/{channel}: {label} has shape "
            f"{array.shape}; expected {expected_shape}."
        )

    if not np.all(np.isfinite(array)):
        raise ValueError(
            f"S{subject:02d}/{channel}: "
            f"{label} contains non-finite values."
        )


loaded_results = {channel: {} for channel in CHANNELS}
pairing_qc_rows = []


for channel in CHANNELS:
    print(f"Auditing channel {channel}")

    for subject in SUBJECTS:
        matched = load_experiment_result(str(file_map[channel][subject]["matched"]))
        shuffled = load_experiment_result(str(file_map[channel][subject]["shuffled"]))

        check_result_metadata(matched, subject, channel, EOG_CONDITION)
        check_result_metadata(shuffled, subject, channel, SHUFFLED_CONDITION)
        check_shuffled_config(shuffled.config)

        matched_fold_ids = list(matched.fold_ids())
        shuffled_fold_ids = list(shuffled.fold_ids())

        if matched_fold_ids != shuffled_fold_ids:
            raise ValueError(
                f"S{subject:02d}/{channel}: "
                "CV fold IDs differ between conditions."
            )

        config = shuffled.config

        if int(config["shuffle_seed"]) != SHUFFLE_SEED:
            raise ValueError(
                f"S{subject:02d}/{channel}: "
                "shuffle seed mismatch."
            )

        for fold_position, fold_idx in enumerate(shuffled_fold_ids):
            mf = matched.folds[fold_idx]
            sf = shuffled.folds[fold_idx]

            # ---------------------------------------------------------
            # Read the shuffled-pairing fold metadata
            # ---------------------------------------------------------

            train = np.asarray(sf["train_trials"], dtype=int).reshape(-1)
            test = np.asarray(sf["test_trials"], dtype=int).reshape(-1)
            train_source = np.asarray(sf["train_eog_source_trials"], dtype=int).reshape(-1)
            test_source = np.asarray(sf["test_eog_source_trials"], dtype=int).reshape(-1)
            train_perm = np.asarray(sf["train_permutation"], dtype=int).reshape(-1)
            test_perm = np.asarray(sf["test_permutation"], dtype=int).reshape(-1)

            # ---------------------------------------------------------
            # Partition sizes and coverage
            # ---------------------------------------------------------

            if len(train) != N_TRAIN_TRIALS:
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "incorrect training partition size."
                )

            if len(test) != N_TEST_TRIALS:
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "incorrect testing partition size."
                )

            if np.intersect1d(train, test).size:
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "training and testing partitions overlap."
                )

            if not np.array_equal(
                np.sort(np.concatenate([train, test])),
                np.arange(N_TRIALS),
            ):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "training/test partitions do not cover all trials."
                )

            # ---------------------------------------------------------
            # Permutation validity
            # ---------------------------------------------------------

            if not np.array_equal(np.sort(train_perm), np.arange(len(train))):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "invalid training permutation."
                )

            if not np.array_equal(np.sort(test_perm), np.arange(len(test))):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "invalid testing permutation."
                )

            # ---------------------------------------------------------
            # Verify source-trial mappings
            # ---------------------------------------------------------

            if not np.array_equal(train_source, train[train_perm]):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "training EOG source mapping is inconsistent."
                )

            if not np.array_equal(test_source, test[test_perm]):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "testing EOG source mapping is inconsistent."
                )

            if not np.array_equal(np.sort(train_source), np.sort(train)):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "training EOG source outside training partition."
                )

            if not np.array_equal(np.sort(test_source), np.sort(test)):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "testing EOG source outside testing partition."
                )

            # ---------------------------------------------------------
            # No fixed points
            # ---------------------------------------------------------

            if np.any(train == train_source):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "training pairing contains a fixed point."
                )

            if np.any(test == test_source):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "testing pairing contains a fixed point."
                )

            # ---------------------------------------------------------
            # Validate deterministic fold-specific seeds
            # ---------------------------------------------------------

            expected_train_seed = SHUFFLE_SEED + fold_position
            expected_test_seed = SHUFFLE_SEED + 1000 + fold_position
            train_seed = int(sf["train_seed"])
            test_seed = int(sf["test_seed"])

            if train_seed != expected_train_seed:
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    f"incorrect training seed {train_seed}; "
                    f"expected {expected_train_seed}."
                )

            if test_seed != expected_test_seed:
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    f"incorrect testing seed {test_seed}; "
                    f"expected {expected_test_seed}."
                )

            # ---------------------------------------------------------
            # Confirm that shuffled predictions retain original targets
            # ---------------------------------------------------------

            if not np.array_equal(np.asarray(sf["trials"], dtype=int), test):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "shuffled result target IDs do not match test trials."
                )

            if not np.array_equal(
                np.asarray(mf["trials"], dtype=int),
                np.asarray(sf["trials"], dtype=int),
            ):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "matched and shuffled target trial IDs differ."
                )

            if not np.allclose(mf["y_test"], sf["y_test"], rtol=0, atol=1e-12):
                raise ValueError(
                    f"S{subject:02d}/{channel}/fold {fold_idx}: "
                    "held-out EEG target values differ between conditions."
                )

            pairing_qc_rows.append({
                "Subject": subject,
                "Channel": channel,
                "Fold": fold_idx,
                "N train": len(train),
                "N test": len(test),
                "Train mapping valid": True,
                "Test mapping valid": True,
                "Train no fixed points": True,
                "Test no fixed points": True,
                "Targets identical": True,
                "Train seed": train_seed,
                "Test seed": test_seed,
            })

        # -------------------------------------------------------------
        # Reconstruct complete trial-level arrays
        # -------------------------------------------------------------

        y_eog, pred_eog = matched.get_all_trials(
            signal_fs=EEG_FS,
            trial_len_sec=TRIAL_DURATION,
        )

        y_shuffled, pred_shuffled = shuffled.get_all_trials(
            signal_fs=EEG_FS,
            trial_len_sec=TRIAL_DURATION,
        )

        y_eog = np.asarray(y_eog)
        pred_eog = np.asarray(pred_eog)
        y_shuffled = np.asarray(y_shuffled)
        pred_shuffled = np.asarray(pred_shuffled)

        ids_eog = np.asarray(matched.get_trial_indices(), dtype=int).reshape(-1)
        ids_shuffled = np.asarray(shuffled.get_trial_indices(), dtype=int).reshape(-1)

        # The project helper verifies alignment before canonical sorting.
        validate_trial_alignment(
            ids_eog,
            ids_shuffled,
            name_a=EOG_CONDITION,
            name_b=SHUFFLED_CONDITION,
        )

        # Canonical order is the original EEG target trial ID.
        ids_eog, y_eog, pred_eog = sort_trials_by_id(ids_eog, y_eog, pred_eog)
        ids_shuffled, y_shuffled, pred_shuffled = sort_trials_by_id(ids_shuffled, y_shuffled, pred_shuffled)

        if not np.array_equal(ids_eog, np.arange(N_TRIALS)):
            raise ValueError(
                f"S{subject:02d}/{channel}: "
                "target trial IDs do not cover 0–39."
            )

        if not np.array_equal(ids_eog, ids_shuffled):
            raise ValueError(
                f"S{subject:02d}/{channel}: "
                "target trial IDs differ after sorting."
            )

        # Check all array dimensions and numerical values.
        check_trial_array(y_eog, "EOG model targets", subject, channel)
        check_trial_array(pred_eog, "EOG model predictions", subject, channel)
        check_trial_array(y_shuffled, "Shuffled targets", subject, channel)
        check_trial_array(pred_shuffled, "Shuffled predictions", subject, channel)

        # Both models must use the same original EEG target trials.
        if not np.allclose(y_eog, y_shuffled, rtol=0, atol=1e-12):
            max_difference = float(np.max(np.abs(y_eog - y_shuffled)))

            raise ValueError(
                f"S{subject:02d}/{channel}: reconstructed EEG targets "
                f"differ between conditions. "
                f"Maximum absolute difference: {max_difference}."
            )

        loaded_results[channel][subject] = {
            "y_true": y_eog,
            "trial_ids": ids_eog,
            "pred": {
                EOG_CONDITION: pred_eog,
                SHUFFLED_CONDITION: pred_shuffled,
            },
        }


df_pairing_qc = pd.DataFrame(pairing_qc_rows)

print("Pairing and target-alignment checks passed.")
print(f"Audited fold records: {len(df_pairing_qc)}")

display(df_pairing_qc.head(10))

## 6. Prediction Variability and Numerical QC

Evaluate the numerical quality and temporal variability of the held-out
predictions.

For each participant and condition, calculate the standard deviation of
each predicted EEG trial and its ratio to the corresponding target EEG
standard deviation.

This is a descriptive quality-control step. It does not replace the
correlation or RMSE analysis, and no trial is excluded solely because
its prediction has low variability.

In [ ]:
# %%
prediction_qc_rows = []

for channel in CHANNELS:
    for subject in SUBJECTS:
        data = loaded_results[channel][subject]
        y_true = data["y_true"]
        target_sd = np.std(y_true, axis=1)

        if np.any(target_sd <= 0):
            raise ValueError(f"S{subject:02d}/{channel}: constant EEG target trial detected.")

        for condition in CONDITIONS:
            y_pred = data["pred"][condition]
            pred_sd = np.std(y_pred, axis=1)
            sd_ratio = pred_sd / target_sd
            prediction_qc_rows.append({
                "Subject": subject,
                "Channel": channel,
                "Condition": condition,
                "Median prediction SD": float(np.median(pred_sd)),
                "Median target SD": float(np.median(target_sd)),
                "Median prediction/target SD ratio": float(np.median(sd_ratio)),
                "Minimum prediction SD": float(np.min(pred_sd)),
                "Zero-variance predictions": int(np.sum(pred_sd == 0)),
                "Near-zero predictions (SD <= 1e-12)": int(np.sum(pred_sd <= 1e-12)),
            })

df_prediction_qc = pd.DataFrame(prediction_qc_rows)

display(df_prediction_qc.round(5))

## 7. Trial-Level Performance Summaries

Calculate trial-level Pearson correlation and RMSE for both conditions
using the project's `summarize_trials()` function.

For each participant, the 40 trial-level correlations are Fisher-z
transformed, averaged with equal trial weights, and back-transformed
to obtain the participant-level pooled correlation.

The same bootstrap configuration is used for both conditions.

In [ ]:
# %%
subject_summaries = {channel: {} for channel in CHANNELS}

for channel in CHANNELS:
    for subject in SUBJECTS:
        data = loaded_results[channel][subject]
        y_true = data["y_true"]
        subject_summaries[channel][subject] = {}
        for condition in CONDITIONS:
            y_pred = data["pred"][condition]
            subject_summaries[channel][subject][condition] = summarize_trials(y_true, y_pred, fs=EEG_FS, seed=SEED, n_boot=N_BOOT, ci_level=CI_LEVEL)

print("Trial-level summaries computed.")
print(f"Conditions   : {len(CONDITIONS)}")
print(f"Participants : {len(SUBJECTS)}")
print(f"Channels     : {len(CHANNELS)}")

## 8. Participant-Level Performance Table

Report the participant-level pooled correlation, its trial-bootstrap
confidence interval, mean trial-level RMSE, and its trial-bootstrap
confidence interval.

These intervals describe within-participant trial variability. They
are not population-level confidence intervals.

In [ ]:
# %%
def ci_bounds(ci):
    """Extract lower and upper confidence limits."""
    if isinstance(ci, dict):
        if "low" in ci and "high" in ci:
            return float(ci["low"]), float(ci["high"])
        if "lower" in ci and "upper" in ci:
            return float(ci["lower"]), float(ci["upper"])
        raise KeyError(f"Unrecognized confidence interval keys: {list(ci)}")

    values = np.asarray(ci, dtype=float).reshape(-1)

    if values.size < 2:
        raise ValueError("A confidence interval must contain two limits.")

    return float(values[0]), float(values[1])


subject_rows = []

for channel in CHANNELS:
    for subject in SUBJECTS:
        for condition in CONDITIONS:
            summary = subject_summaries[channel][subject][condition]

            r_low, r_high = ci_bounds(summary["r_pooled_ci_boot"])
            rmse_low, rmse_high = ci_bounds(summary["rmse_trial_ci_boot"])

            r_trials = np.asarray(summary["r_trials"], dtype=float)
            rmse_trials = np.asarray(summary["rmse_trials"], dtype=float)

            subject_rows.append({
                "Subject": subject,
                "Channel": channel,
                "Condition": condition,
                "N trials": summary["trials"],
                "Defined correlations": int(np.isfinite(r_trials).sum()),
                "Undefined correlations": int(np.isnan(r_trials).sum()),
                "Defined RMSE": int(np.isfinite(rmse_trials).sum()),
                "Pooled r": summary["r_pooled"],
                "Pooled r trial-CI low": r_low,
                "Pooled r trial-CI high": r_high,
                "Mean trial RMSE": summary["rmse_mean"],
                "RMSE trial-CI low": rmse_low,
                "RMSE trial-CI high": rmse_high,
            })


df_subject = pd.DataFrame(subject_rows)

display(df_subject.sort_values(["Channel", "Subject", "Condition"]).round(3))

## 9. Within-Participant Paired Comparisons

Compare EOG model and shuffled-pairing predictions within each participant
using the same original held-out EEG target trials.

The paired comparisons use the project's `compare_methods_trial_level()`
function.

Effect directions:

- Δz(r) = EOG model − shuffled-pairing model.
  Positive values favor the EOG model.
- ΔRMSE = EOG model − shuffled-pairing model.
  Negative values favor the EOG model.

These comparisons are conducted separately within each participant.
Population-level inference is performed in the following sections,
using participants as independent units.

In [ ]:
# %%
within_subject_tests = {channel: {} for channel in CHANNELS}
within_rows = []

for channel in CHANNELS:
    for subject in SUBJECTS:
        eog_summary = subject_summaries[channel][subject][EOG_CONDITION]
        shuffled_summary = subject_summaries[channel][subject][SHUFFLED_CONDITION]

        comparison = compare_methods_trial_level(
            eog_summary, shuffled_summary,
            method_names=(EOG_CONDITION, SHUFFLED_CONDITION),
            seed=SEED, n_perm=N_PERM, n_boot=N_BOOT,
        )

        within_subject_tests[channel][subject] = comparison

        r_summary = comparison["r"]["summary"]
        rmse_summary = comparison["rmse"]["summary"]

        r_low, r_high = ci_bounds(r_summary["ci_boot"])
        rmse_low, rmse_high = ci_bounds(rmse_summary["ci_boot"])

        within_rows.append({
            "Subject": subject,
            "Channel": channel,
            "Comparison": "EOG model - Shuffled-pairing model, original-target scoring",
            "Delta z(r)": r_summary["mean"],
            "Delta z(r) CI low": r_low,
            "Delta z(r) CI high": r_high,
            "Delta z(r) sign-flip p": r_summary["tests"]["sign_perm_p"],
            "Delta r": comparison["r"].get("delta_mean_r", np.nan),
            "Delta RMSE": rmse_summary["mean"],
            "Delta RMSE CI low": rmse_low,
            "Delta RMSE CI high": rmse_high,
            "Delta RMSE sign-flip p": rmse_summary["tests"]["sign_perm_p"],
        })


df_within_subject = pd.DataFrame(within_rows)

display(df_within_subject.sort_values(["Channel", "Subject"]).round(3))

## 10. Population-Level Aggregation and Inference

Population summaries use participants as the independent units.

Correlation estimates and their confidence intervals are calculated on
the Fisher-z scale and back-transformed for presentation.

RMSE estimates and confidence intervals are calculated on the original
RMSE scale.

For paired comparisons, participant-level effects are tested using
sign-flip permutation tests and Wilcoxon signed-rank tests.

In [ ]:
# %%
def aggregate_population(values, scale="linear", n_boot=N_BOOT, ci_level=CI_LEVEL, seed=SEED):
    """
    Aggregate participant-level values and calculate a bootstrap CI.

    The input values must represent independent participants, not
    individual trials.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = values.size

    if n == 0:
        return {"n": 0, "mean": np.nan, "ci_lo": np.nan, "ci_hi": np.nan}

    transformed = fisher_z(values) if scale == "fisher" else values
    mean_transformed = float(np.mean(transformed))

    if n > 1:
        rng = np.random.default_rng(seed)
        bootstrap = stats.bootstrap(
            (transformed,),
            np.mean,
            confidence_level=ci_level,
            n_resamples=n_boot,
            method="percentile",
            random_state=rng,
        )
        ci_lo = float(bootstrap.confidence_interval.low)
        ci_hi = float(bootstrap.confidence_interval.high)
    else:
        ci_lo = mean_transformed
        ci_hi = mean_transformed

    if scale == "fisher":
        mean_display = float(inv_fisher_z(mean_transformed))
        ci_lo_display = float(inv_fisher_z(ci_lo))
        ci_hi_display = float(inv_fisher_z(ci_hi))
    else:
        mean_display = mean_transformed
        ci_lo_display = ci_lo
        ci_hi_display = ci_hi

    return {"n": n, "mean": mean_display, "ci_lo": ci_lo_display, "ci_hi": ci_hi_display}


def effect_stats(values, scale="linear", n_perm=N_PERM, seed=SEED):
    """
    Summarize and test participant-level effects.

    For scale='fisher', input values are participant-level correlations.
    The values are Fisher-z transformed before the population test.

    For scale='linear', input values are already on their analysis scale,
    for example paired differences in Fisher-z correlation or RMSE.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        return {
            "N": 0, "mean": np.nan, "CI_low": np.nan, "CI_high": np.nan,
            "signflip_p": np.nan, "wilcoxon_p": np.nan, "Cohen_dz": np.nan,
        }

    aggregate = aggregate_population(values, scale=scale, n_boot=N_BOOT, ci_level=CI_LEVEL, seed=seed)

    test_values = fisher_z(values) if scale == "fisher" else values

    p_signflip = float(sign_flip_pvalue(test_values, n_perm=n_perm, seed=seed))

    try:
        p_wilcoxon = float(
            stats.wilcoxon(test_values, alternative="two-sided", zero_method="wilcox").pvalue
        )
    except (ValueError, ZeroDivisionError):
        p_wilcoxon = np.nan

    sd = float(np.std(test_values, ddof=1)) if test_values.size > 1 else np.nan
    dz = float(np.mean(test_values) / sd) if np.isfinite(sd) and sd > 0 else np.nan

    return {
        "N": aggregate["n"],
        "mean": aggregate["mean"],
        "CI_low": aggregate["ci_lo"],
        "CI_high": aggregate["ci_hi"],
        "signflip_p": p_signflip,
        "wilcoxon_p": p_wilcoxon,
        "Cohen_dz": dz,
    }

## 11. Population-Level Condition Summary

Estimate the population-level correlation and RMSE separately for the
EOG model and shuffled-pairing model.

For correlation, participant-level pooled correlations are Fisher-z
transformed before population aggregation.

A one-sample sign-flip test and Wilcoxon signed-rank test assess whether
the population-level correlation differs from zero. These tests use
participant-level estimates, not the 40 trials as independent
observations.

The correlation tests are Holm-adjusted across the three EEG channels
separately for each condition.

In [ ]:
# %%
population_condition_rows = []

for channel in CHANNELS:
    for condition in CONDITIONS:
        r_values = np.array([
            subject_summaries[channel][subject][condition]["r_pooled"]
            for subject in SUBJECTS
        ], dtype=float)

        rmse_values = np.array([
            subject_summaries[channel][subject][condition]["rmse_mean"]
            for subject in SUBJECTS
        ], dtype=float)

        # Population correlation, including test against zero.
        r_summary = effect_stats(r_values, scale="fisher", n_perm=N_PERM, seed=SEED)

        # Population RMSE summary.
        rmse_summary = aggregate_population(rmse_values, scale="linear", n_boot=N_BOOT, ci_level=CI_LEVEL, seed=SEED)

        population_condition_rows.append({
            "Channel": channel,
            "Condition": condition,
            "N participants (r)": r_summary["N"],
            "Population r": r_summary["mean"],
            "r 95% CI low": r_summary["CI_low"],
            "r 95% CI high": r_summary["CI_high"],
            "r sign-flip p vs 0": r_summary["signflip_p"],
            "r Wilcoxon p vs 0": r_summary["wilcoxon_p"],
            "r Cohen dz vs 0": r_summary["Cohen_dz"],
            "N participants (RMSE)": rmse_summary["n"],
            "Population RMSE": rmse_summary["mean"],
            "RMSE 95% CI low": rmse_summary["ci_lo"],
            "RMSE 95% CI high": rmse_summary["ci_hi"],
        })


df_population_conditions = pd.DataFrame(population_condition_rows)


# Holm correction across the three channels, separately for each condition.
df_population_conditions["r Holm p"] = np.nan

for condition in CONDITIONS:
    mask = df_population_conditions["Condition"].eq(condition)
    p_values = df_population_conditions.loc[mask, "r sign-flip p vs 0"].to_numpy(dtype=float)
    valid = np.isfinite(p_values)
    adjusted = np.full(p_values.shape, np.nan)

    if valid.any():
        adjusted[valid] = multipletests(p_values[valid], method="holm")[1]

    df_population_conditions.loc[mask, "r Holm p"] = adjusted


display(df_population_conditions.sort_values(["Channel", "Condition"]).round(3))

## 12. Population-Level EOG-versus-Shuffled Effects

Aggregate the within-participant paired effects across participants.

For correlation, participant-level effects are differences in trial-level
Fisher-z correlation.

For RMSE, participant-level effects are differences in mean trial-level
RMSE.

The effect definitions are:

- Δz(r) = EOG model − shuffled-pairing model.
- ΔRMSE = EOG model − shuffled-pairing model.

Positive Δz(r) favors the EOG model. Negative ΔRMSE favors the EOG model.

The sign-flip test and Wilcoxon signed-rank test are conducted across
participants. Holm correction is applied across the three EEG channels
separately for correlation and RMSE.

In [ ]:
# %%
population_effect_rows = []

for channel in CHANNELS:
    # One paired effect per participant.
    delta_z = np.array([
        within_subject_tests[channel][subject]["r"]["summary"]["mean"]
        for subject in SUBJECTS
    ], dtype=float)

    delta_rmse = np.array([
        within_subject_tests[channel][subject]["rmse"]["summary"]["mean"]
        for subject in SUBJECTS
    ], dtype=float)

    # The correlation effect is already expressed in Fisher-z units.
    # Do not apply Fisher-z transformation to the difference again.
    r_effect = effect_stats(delta_z, scale="linear", n_perm=N_PERM, seed=SEED)

    # RMSE differences are on the original RMSE scale.
    rmse_effect = effect_stats(delta_rmse, scale="linear", n_perm=N_PERM, seed=SEED)

    population_effect_rows.extend([
        {
            "Channel": channel,
            "Metric": "Delta z(r)",
            "Contrast": "EOG model - Shuffled-pairing model, original-target scoring",
            **r_effect,
        },
        {
            "Channel": channel,
            "Metric": "Delta RMSE",
            "Contrast": "EOG model - Shuffled-pairing model, original-target scoring",
            **rmse_effect,
        },
    ])


df_population_effects = pd.DataFrame(population_effect_rows)

df_population_effects.rename(
    columns={"signflip_p": "Sign-flip p", "wilcoxon_p": "Wilcoxon p", "Cohen_dz": "Cohen dz"},
    inplace=True,
)


# Holm correction across the three channels, separately for each metric.
df_population_effects["Holm-adjusted p"] = np.nan

for metric in ("Delta z(r)", "Delta RMSE"):
    mask = df_population_effects["Metric"].eq(metric)
    p_values = df_population_effects.loc[mask, "Sign-flip p"].to_numpy(dtype=float)
    valid = np.isfinite(p_values)
    adjusted = np.full(p_values.shape, np.nan)

    if valid.any():
        adjusted[valid] = multipletests(p_values[valid], method="holm")[1]

    df_population_effects.loc[mask, "Holm-adjusted p"] = adjusted


display(df_population_effects.sort_values(["Metric", "Channel"]).round(3))

## 13. EOG Model versus Shuffled-Pairing Figures

Create paired participant-level plots for pooled correlation and RMSE.

Each line represents one participant. Population-level markers and
error bars represent the participant-aggregated estimate and its
95% bootstrap confidence interval.

The three figures correspond to Fz, Pz, and Oz.

The plotted conditions are:

- EOG model
- Shuffled (orig. target)

In [ ]:
fig, axes = plot_eog_shuffled_pairing_summary(
    df_subject=df_subject,
    df_group=df_population_conditions,
    subjects=SUBJECTS,
    channels=CHANNELS,
    save_fig=True,
    file_name=FIG_DIR / "eog_shuffled_pairing_null_models_all_channels.pdf",
    seed=SEED,
)
plt.show()
plt.close(fig)

## 14. Export Analysis Tables

Export the pairing QC, prediction variability QC, participant-level
summaries, within-participant paired comparisons, population condition
summaries, and population-level paired effects.

All output filenames use the `eog_shuffled_pairing` terminology.

In [ ]:
# %%
exports = {
    "eog_shuffled_pairing_qc.csv": df_pairing_qc,
    "eog_shuffled_pairing_prediction_qc.csv": df_prediction_qc,
    "eog_shuffled_pairing_subject_performance.csv": df_subject,
    "eog_shuffled_pairing_within_subject_effects.csv": df_within_subject,
    "eog_shuffled_pairing_population_conditions.csv": df_population_conditions,
    "eog_shuffled_pairing_population_effects.csv": df_population_effects,
}

for filename, dataframe in exports.items():
    output_path = TABLE_DIR / filename
    dataframe.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

print("\nAnalysis completed.")
print("Figures:", FIG_DIR)
print("Tables :", TABLE_DIR)

In [ ]:
# %%
print("=" * 80)
print("POPULATION CONDITION SUMMARIES")
print("=" * 80)

display(
    df_population_conditions[[
        "Channel", "Condition", "N participants (r)", "Population r",
        "r 95% CI low", "r 95% CI high", "N participants (RMSE)",
        "Population RMSE", "RMSE 95% CI low", "RMSE 95% CI high",
    ]]
    .sort_values(["Channel", "Condition"])
    .round(3)
)


print("=" * 80)
print("POPULATION PAIRED EFFECTS")
print("EOG model minus shuffled-pairing model")
print("=" * 80)

display(
    df_population_effects[[
        "Channel", "Metric", "N", "mean", "CI_low", "CI_high",
        "Sign-flip p", "Wilcoxon p", "Holm-adjusted p", "Cohen dz",
    ]]
    .sort_values(["Metric", "Channel"])
    .round(3)
)